# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/the-lazyguy/ML-flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [21]:
import numpy as np
import pandas as pd

def compute_baseline_action_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes baseline action score (0-100) and assigns primary reason codes.
    """
    df = df.copy()

    # 1. Component calculations
    # Traffic decay component
    traffic_decay = np.maximum(0, (df['prior_impressions'] - df['current_clicks']) / np.maximum(df['prior_impressions'], 1))

    # CTR Gap component
    observed_ctr = df['current_clicks'] / np.maximum(df['current_impressions'], 1)
    ctr_gap = np.maximum(0, df['expected_ctr'] - observed_ctr)

    # Staleness component
    staleness = np.minimum(1.0, df['days_since_edit'] / 365.0)

    # Composite score calculation (0 to 100)
    raw_score = (0.40 * traffic_decay + 0.35 * ctr_gap * 10 + 0.25 * staleness) * 100
    df['action_score'] = np.clip(raw_score, 0, 100).round(1)

    # 2. Assign Reason Codes
    conditions = [
        (traffic_decay > 0.35),
        (ctr_gap > 0.03) & (df['current_impressions'] > df['current_impressions'].median()),
        (df['days_since_edit'] > 180),
        (df['word_count'] < 400)
    ]
    choices = [
        'RC_HIGH_DECAY',
        'RC_LOW_CTR_HIGH_IMP',
        'RC_STALE_HIGH_POTENTIAL',
        'RC_THIN_CONTENT_GAP'
    ]

    df['reason_code'] = np.select(conditions, choices, default='RC_ROUTINE_MAINTENANCE')
    return df

print("✓ Baseline action score rule definition and reason code mapper ready.")

✓ Baseline action score rule definition and reason code mapper ready.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
import os

def build_and_export_ranked_queue(num_samples: int = 100) -> pd.DataFrame:
    """
    Generates dataset, calculates action scores, ranks pages, and writes to CSV.
    """
    np.random.seed(42)

    # Generate representative page performance dataset
    data = {
        'page_id': [f"page_{i:04d}" for i in range(1, num_samples + 1)],
        'word_count': np.random.randint(200, 2500, size=num_samples),
        'days_since_edit': np.random.randint(10, 500, size=num_samples),
        'prior_impressions': np.random.randint(1000, 50000, size=num_samples),
        'current_impressions': np.random.randint(800, 48000, size=num_samples),
        'current_clicks': np.random.randint(20, 2000, size=num_samples),
        'expected_ctr': np.random.uniform(0.02, 0.08, size=num_samples)
    }

    df_raw = pd.DataFrame(data)

    # Apply baseline rule
    df_scored = compute_baseline_action_score(df_raw)

    # Sort descending by action score
    df_ranked = df_scored.sort_values(by='action_score', ascending=False).reset_index(drop=True)
    df_ranked['rank'] = df_ranked.index + 1

    # Reorder columns
    cols_order = ['rank', 'page_id', 'action_score', 'reason_code', 'word_count',
                  'days_since_edit', 'current_impressions', 'current_clicks']
    df_ranked = df_ranked[cols_order]

    # Ensure export directory exists and save CSV
    output_dir = 'work/outputs'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'baseline_action_score.csv')
    df_ranked.to_csv(output_path, index=False)

    print(f"✓ Ranked queue built successfully. Output written to: {output_path}")
    print(f"Total pages scored: {len(df_ranked)}")
    return df_ranked

# Execute queue build
df_queue = build_and_export_ranked_queue()

✓ Ranked queue built successfully. Output written to: work/outputs/baseline_action_score.csv
Total pages scored: 100


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [23]:
# Display top 20 items from exported queue
top_20 = df_queue.head(20)
print("=== TOP 20 ACTION QUEUE REVIEW ===")
print(top_20[['rank', 'page_id', 'action_score', 'reason_code', 'word_count', 'days_since_edit']].to_string(index=False))

=== TOP 20 ACTION QUEUE REVIEW ===
 rank   page_id  action_score   reason_code  word_count  days_since_edit
    1 page_0065          83.6 RC_HIGH_DECAY         991              368
    2 page_0051          82.0 RC_HIGH_DECAY        1700              357
    3 page_0046          79.9 RC_HIGH_DECAY        2453              349
    4 page_0067          79.4 RC_HIGH_DECAY         963              465
    5 page_0037          78.0 RC_HIGH_DECAY        2261              179
    6 page_0010          76.0 RC_HIGH_DECAY        1682              389
    7 page_0058          75.6 RC_HIGH_DECAY        1015              292
    8 page_0019          75.5 RC_HIGH_DECAY         659              397
    9 page_0001          75.3 RC_HIGH_DECAY        1060              264
   10 page_0073          75.3 RC_HIGH_DECAY         264              316
   11 page_0048          73.8 RC_HIGH_DECAY        1785              455
   12 page_0031          73.4 RC_HIGH_DECAY        1728              336
   13 page_0059 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [24]:
def audit_leakage_and_weak_picks(df: pd.DataFrame):
    """
    Asserts zero leakage and checks distribution of weak picks.
    """
    print("=== LEAKAGE & QUALITY ASSURANCE AUDIT ===\n")

    # 1. Check for prohibited future metrics or direct targets in input columns
    forbidden_cols = ['post_action_conversion', 'future_clicks_30d', 'user_id', 'client_id']
    detected_forbidden = [col for col in forbidden_cols if col in df.columns]

    assert len(detected_forbidden) == 0, f"LEAKAGE DETECTED: Found forbidden columns {detected_forbidden}"
    print("✓ Leakage Audit Passed: Zero post-cutoff or forbidden target columns present.")

    # 2. Check score distribution properties
    scores = df['action_score']
    assert scores.min() >= 0 and scores.max() <= 100, "SCORE RANGE ERROR: Scores out of bounds [0, 100]"
    print(f"✓ Score Distribution Valid: Min = {scores.min()}, Max = {scores.max()}, Median = {scores.median():.1f}")

    # 3. Highlight potential weak pick candidates (Thin content with high current clicks)
    weak_picks = df[(df['reason_code'] == 'RC_THIN_CONTENT_GAP') & (df['current_clicks'] > df['current_clicks'].median())]
    print(f"✓ Weak Pick Audit: Identified {len(weak_picks)} thin content pages that still perform well (candidates for exception rules).")

# Run audit
audit_leakage_and_weak_picks(df_queue)

=== LEAKAGE & QUALITY ASSURANCE AUDIT ===

✓ Leakage Audit Passed: Zero post-cutoff or forbidden target columns present.
✓ Score Distribution Valid: Min = 25.5, Max = 83.6, Median = 56.8
✓ Weak Pick Audit: Identified 0 thin content pages that still perform well (candidates for exception rules).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.